# Step 4: Upload Data to S3

**Purpose**: Upload the generated machine temperature CSV to S3 bucket

**Prerequisites**:
- Run `01_setup_env.ipynb` to create `.env` configuration
- Run `02_create_s3_bucket.ipynb` to create the S3 bucket
- Run `03_generate_data.ipynb` to generate `data/machines.csv`

## Load Configuration

In [ ]:
import boto3
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')
region = os.getenv('REGION')

print(f"Bucket: {bucket_name}")
print(f"Region: {region}")

# Initialize S3 client
s3 = boto3.client('s3', region_name=region)

## Verify Local Data File Exists

In [ ]:
# Path to the generated CSV file
local_file = 'data/machines.csv'

if os.path.exists(local_file):
    file_size = os.path.getsize(local_file)
    print(f"Found: {local_file}")
    print(f"Size: {file_size:,} bytes ({file_size / 1024 / 1024:.2f} MB)")
else:
    print(f"File not found: {local_file}")
    print("Please run 0002_generate_data.ipynb first to generate the data.")

## Upload to S3

In [ ]:
# S3 destination path
s3_key = 'data/raw/machines.csv'

print(f"Uploading {local_file} to s3://{bucket_name}/{s3_key}...")

try:
    s3.upload_file(local_file, bucket_name, s3_key)
    print(f"\nUpload successful!")
    print(f"\nS3 URI: s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"Upload failed: {e}")

## Verify Upload

In [ ]:
# Verify the file exists in S3
try:
    response = s3.head_object(Bucket=bucket_name, Key=s3_key)
    s3_size = response['ContentLength']
    last_modified = response['LastModified']
    
    print(f"File verified in S3")
    print(f"  Size: {s3_size:,} bytes")
    print(f"  Last modified: {last_modified}")
    print(f"  S3 URI: s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"Verification failed: {e}")

## List All Files in data/raw/

In [ ]:
# List all files in the data/raw/ prefix
response = s3.list_objects_v2(Bucket=bucket_name, Prefix='data/raw/')

print(f"Files in s3://{bucket_name}/data/raw/:\n")
for obj in response.get('Contents', []):
    print(f"  {obj['Key']} ({obj['Size']:,} bytes)")

if not response.get('Contents'):
    print("  (no files found)")